In [2]:
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import math
import json

In [3]:
dir_processed = Path('../data/processed')
dir_processed.mkdir(exist_ok=True)
dir_raw = Path('../data/raw')
dir_raw.mkdir(exist_ok=True)

In [4]:
!pwd

/files/assignments/final-project/notebooks


In [5]:
def load_json(file):

    with open(file, "r") as f:
        data_raw = json.load(f)

    return pd.DataFrame(data_raw['data'])

In [6]:
c = 0 
full_dataset = []

for file in sorted(dir_raw.glob("*.json")):
    imported_data = load_json(file)

    full_dataset.append(imported_data)

dataset = pd.concat(full_dataset, ignore_index=True)

In [ ]:
dataset['date'] = pd.to_datetime(dataset['event_date'])
type(dataset['date'][1])

In [ ]:
"check what is in the data"

type(dataset)
type(dataset['event_date'])
type(dataset['date'][1])
dataset

In [ ]:
dataset = dataset.sort_values(by='date')
dataset

In [10]:
def df_index(dataframe):
    rows = len(dataframe)

    dataframe.index = range(1, rows+1)
    
    return dataframe

dataset = df_index(dataset)

In [11]:
"from lecture W03D02: missing values appear as an empty string so isna() and isnul() doesn't work"

def missing_values(column_name):
    sum_nas = float((dataset[column_name] == '').sum())
    # print(sum_nas)
    ratio_nas = round(float((dataset[column_name] == '').sum() / len(dataset)), 2)
    # print(ratio_nas)
    return {f"column": {column_name}, "sum": sum_nas, "ratio": ratio_nas}

# dataset['assoc_actor_1'].iloc[0]

In [12]:
"testing function with one column"

missing_values('inter1')

{'column': {'inter1'}, 'sum': 0.0, 'ratio': 0.0}

In [13]:
"finding the column names"

columns = dataset.keys()

print(columns)

Index(['event_id_cnty', 'event_date', 'time_precision', 'disorder_type',
       'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1',
       'actor2', 'assoc_actor_2', 'inter2', 'civilian_targeting', 'country',
       'admin1', 'admin3', 'location', 'geo_precision', 'source', 'fatalities',
       'tags', 'date'],
      dtype='str')


In [14]:
"copying column names over the columns variable"

columns = ['event_id_cnty', 'event_date', 'time_precision', 'disorder_type',
       'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1',
       'actor2', 'assoc_actor_2', 'inter2', 'civilian_targeting', 'country',
       'admin1', 'admin3', 'location', 'geo_precision', 'source', 'fatalities',
       'tags']

In [ ]:
all_missing_values = []

for column in columns:
    print(f"column: {column}")
    missing_value = missing_values(column)
    if missing_value['sum'] > 0:
        all_missing_values.append(missing_value)

    # print(f"missing values: {missing_value}")
    # dict_missing_values[column] = missing_value

In [16]:
len(all_missing_values)
all_missing_values

[{'column': {'assoc_actor_1'}, 'sum': 14198.0, 'ratio': 0.84},
 {'column': {'actor2'}, 'sum': 5356.0, 'ratio': 0.32},
 {'column': {'assoc_actor_2'}, 'sum': 13603.0, 'ratio': 0.81},
 {'column': {'inter2'}, 'sum': 5356.0, 'ratio': 0.32},
 {'column': {'civilian_targeting'}, 'sum': 11645.0, 'ratio': 0.69},
 {'column': {'tags'}, 'sum': 14138.0, 'ratio': 0.84}]

In [30]:
"optional, manual tests to see if other columns contain missing values coded as n/a or null"

dataset['admin3'].isna().all()
dataset['admin3'].isnull().all()

np.False_

In [18]:
"check whether there are duplicate values in dataset"

"duplicated() checks if the row is a duplicate of the former row"

duplicates = dataset[dataset.duplicated()]
duplicates

,event_id_cnty,event_date,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,...,civilian_targeting,country,admin1,admin3,location,geo_precision,source,fatalities,tags,date


In [19]:
# transfer_date = pd.to_datetime('2024-12-08')
# month = timedelta(days=30)
# c = 0
# months_list = []

# for date in dataset['date']:
#     interval = date - transfer_date
#     if date == None:

#         months_list.append('NA')

#     else:
        
#         months = math.floor(interval / month)
#         months_list.append(months)
    
#     c += 1
#     print(f"iteration: {c}: {months}")

# dataset['interval'] = pd.Series(months_list) "Q series or list?"

In [20]:
"setting up parameters to create helper column 'interval', measuring months elapsed since the regime fall"

transfer_date = pd.to_datetime('2024-12-08')
reference_date = pd.to_datetime('2024-12-07')
month = timedelta(days=31)
one_day = timedelta(days=1)

dataset['interval'] = (dataset['date'] - transfer_date).floordiv(month, fill_value=0)

In [21]:
"isin() checks where there are potential missing values in newly created helper column"

value = ['NA']
dataset[dataset['interval'].isin(value)]

,event_id_cnty,event_date,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,actor2,...,country,admin1,admin3,location,geo_precision,source,fatalities,tags,date,interval


In [22]:
"read the dataset back into python if restarting"

# dataset = pd.read_csv(dir_processed / 'dataset.csv')
# dataset.index = dataset.index + 1
# dataset

'read the dataset back into python if restarting'

In [23]:
dataset['start date'] = (transfer_date + dataset['interval'] * month)
# discard: dataset = dataset.rename(columns={'overlay dates':'start date'})

In [ ]:
dataset['end date'] = (dataset['start date'] + month - one_day)

dataset.columns.tolist()

In [ ]:
"check boundaries of individual intervals and whether they match event date" 
"by looking at head and tail ends of returned df"

dataset[dataset['interval'] == 18]

In [28]:
dataset.to_csv(dir_processed/"dataset.csv", index=False)

In [ ]:
# dataset[~dataset['actor2'].isna()]


In [ ]:
# df_post_transfer = df[(df['date'] > '2024-12-08')]
# rows = len(df_post_transfer)
df_post_transfer.index = range(1,rows+1)

In [ ]:
# df_transfer = df[(df['date'] == '2024-12-08')]
# df_transfer_month = df[(df['date'] >= '2024-12-08') & (df['date'] <= '2025-01-07')]
# df_post_transfer = df[(df['date'] > '2024-12-08')]
# df_second_month = df[(df['date'] >= '2025-01-08') & (df['date'] <= '2025-02-07')]


In [ ]:
# df_filtered_by_time = [df_transfer, df_transfer_month, df_post_transfer]

In [ ]:
# df_indexed = [df_index(df) for df in df_filtered_by_time]
# df_transfer
# df_transfer_month
# df_post_transfer

In [ ]:
"create column indicating time elapsed since transfer (month one, two, etc.)"

"use datetime, timedelta"

"round() rounds the float number to two decimals"

# month = round((float(365 / 12)), 2)

# transfer_date = pd.to_datetime('2024-12-08')
# type(transfer_date)

# pd.to_datetime(df['date'])

# -------

# df['date'] = pd.to_datetime(df['date'])


# for date in df['date']:
#     interval = date - transfer_date 
#     if interval <= month:
#         interval_i = 0
#         df_updated = df.assign(period = interval_i)

#     elif interval > month:
#         months = math.floor(interval / month)
#         print(months)
#         df_updated = df.assign(period = months)
#         c += 1
#         print(f"iteration: {c}")
   

#         print(interval_divided)
    # interval = date - transfer_date
    # df['interval'] = df[interval]

   

 

In [ ]:
# df = df.drop(['index', 'level_0'], axis=1)
# df